# Cluster selection — Phase 1 (GBDT)**Problem.** Vertex-timing error in Z+jets is dominated by *cluster selection*, not clustering:a cluster within 60 ps of truth exists in ~87% of events, but the best hand-made score(TRKPTZ) picks it only ~62% of the time. Hand-designed reweightings of the WAVeS jet term(LOJO / JETCAP / KERNEL) all failed — LOJO changed the decision on ~4700 recoverable eventsand was right 47.7% of the time, i.e. a coin flip. So we learn the selector instead.**Framing.** This is *ranking within an event*, not per-cluster classification. That distinctionmatters: an earlier attempt at absolute per-cluster "is this cluster trustworthy?" failed, becausea confidently-wrong cluster looks identical to a right one in reco-only features. Relativecomparison inside one event is a different — and easier — question.**Constraint.** No `lightgbm`/`xgboost` in this kernel, so there is no listwise objective available.We use `HistGradientBoosting` (NaN-native) with **within-event context features** to inject therelative information, and take argmax/argmin *within event* at inference. Three formulations:| | target | pick ||---|---|---|| **A** | binary: is the truth-closest cluster | argmax proba || **B** | binary: within 60 ps of truth | argmax proba || **C** | regress log1p(&#124;Δt&#124;) | **argmin** prediction |**Figure of merit is the physics metric, not AUC**: core fraction (selected cluster within±60 ps of truth), reported against the TRKPTZ baseline and the oracle ceiling.

## 1. Setup

In [ ]:
import glob, os, warningsimport numpy as np, pandas as pd, uprootimport matplotlib.pyplot as pltfrom sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressorfrom sklearn.inspection import permutation_importancewarnings.filterwarnings("ignore")# --- where the exporter's ROOT files live -----------------------------------# condor writes <repo>/<sample>/<sample>_training.root; the local run writes# figs/hists/training.root. Adjust SEARCH_DIRS if you staged them elsewhere.SEARCH_DIRS = [".", "..", "../..", "figs/hists", "../figs/hists"]SAMPLES     = ["vbf", "zjets", "dijet"]PASS_PS   = 60.0     # the physics window (PASS_SIGMA in clustering_constants.h)EVT       = ["sample_id", "event_num"]   # ranking group keyRANDOM    = 0SAMPLE_NAME = {0.0: "vbf", 1.0: "zjets", 2.0: "dijet", 3.0: "local"}def find_files():    hits = {}    for d in SEARCH_DIRS:        for s in SAMPLES:            for p in glob.glob(os.path.join(d, s, f"{s}_training.root")) + \                     glob.glob(os.path.join(d, f"{s}_training.root")):                hits.setdefault(s, os.path.abspath(p))        for p in glob.glob(os.path.join(d, "training.root")):            hits.setdefault("local", os.path.abspath(p))    return hitsFILES = find_files()for k, v in FILES.items():    print(f"  found {k:6s} -> {v}")if not FILES:    print("\\n!! No training ROOT files found. Run:  ./export_training_data --sample=<s>")

## 2. Load

In [ ]:
def load(files):    parts = []    for name, path in files.items():        df = uproot.open(path)["clusters"].arrays(library="pd")        parts.append(df)        print(f"  {name:6s} {len(df):>8,} cluster rows   "              f"{df.groupby(EVT).ngroups:>7,} events   ({path})")    return pd.concat(parts, ignore_index=True)df = load(FILES)df["abs_dt"] = df["delta_t"].abs()print(f"\ntotal {len(df):,} rows / {df.groupby(EVT).ngroups:,} events / {df.shape[1]} columns")

## 3. Labels and baselinesLabels come from `delta_t` (cluster time − truth HS time), which is the **target**, never a feature.`is_best` = the truth-closest cluster in the event; `within60` = inside the physics window(multi-positive — several clusters can qualify).

In [ ]:
g = df.groupby(EVT, sort=False)["abs_dt"]df["is_best"]  = (df["abs_dt"] == g.transform("min")).astype(int)df["within60"] = (df["abs_dt"] < PASS_PS).astype(int)# an event is RECOVERABLE if any cluster is inside the window; a "win" on an# unrecoverable event is luck, so we report both inclusive and conditional numbersdf["recoverable"] = (g.transform("min") < PASS_PS).astype(int)def select_metric(frame, col, higher_is_better=True):    """Pick one cluster per event by `col`, return (core fraction, median |dt|)."""    s = frame[col]    s = s.fillna(-np.inf if higher_is_better else np.inf)    idx = s.groupby([frame[c] for c in EVT], sort=False)    idx = idx.idxmax() if higher_is_better else idx.idxmin()    sel = frame.loc[idx.to_numpy()]    return 100.0 * (sel["abs_dt"] < PASS_PS).mean(), sel["abs_dt"].median()def report(frame, rows, title):    print(f"\n{title}   ({frame.groupby(EVT).ngroups:,} events)")    print(f"  {'selector':26s} {'core<60ps':>10s} {'median|dt|':>11s}")    print("  " + "-" * 50)    for name, col, hib in rows:        cf, md_ = select_metric(frame, col, hib)        print(f"  {name:26s} {cf:9.1f}% {md_:10.1f} ps")BASELINES = [("oracle (truth-closest)", "abs_dt", False),             ("TRKPTZ", "trkptz_score", True),             ("WAVeS",  "waves_score",  True),             ("sumpt",  "sumpt",        True)]report(df, BASELINES, "ALL SAMPLES")for sid, sub in df.groupby("sample_id"):    report(sub, BASELINES, f"sample = {SAMPLE_NAME.get(sid, sid)}")

## 4. FeaturesThree things happen here:1. **Leakage guard** — every `truth_*` column, `delta_t`/`abs_dt` and the derived labels are   dropped from `X`. `sample_id` is dropped too, so the cross-topology test in §8 is honest.2. **Degenerate columns** are detected and removed automatically (all-NaN or constant). This   catches the known ones without hard-coding: `frac_valid_time`/`n_valid_time`/`sumpt_valid_time`   are degenerate by construction (clustering already requires valid times), `mean/min_quality`   is constant in these samples, and the lepton block is empty outside Z+jets.3. **Within-event normalization** — the load-bearing step given we have no listwise loss. For a   curated set of features we add `_ratio_to_max` and `_rank` *within the event*, which is what   lets a per-cluster model express "best of what's available here".

In [ ]:
LEAK = [c for c in df.columns if c.startswith("truth_")] + \       ["delta_t", "abs_dt", "is_best", "within60", "recoverable"]IDS  = ["event_num", "cluster_idx", "sample_id", "weight"]# Curated list for within-event normalization: continuous, physically meaningful,# and plausibly comparative. (Normalizing all ~90 would triple the width for little gain.)NORMALIZE = ["sumpt", "sumpt2", "maxpt", "n_tracks", "cluster_time_sigma",             "time_chi2_ndf", "delta_z_resunits", "z_chi2_ndf", "mean_nhgtd",             "trkptz_score", "waves_score", "frac_pt_in_fwdjet", "lead_pt_frac",             "dz_to_lepton_signif", "sumpt_in_fwdjet"]def add_event_context(frame):    frame = frame.copy()    gb = frame.groupby(EVT, sort=False)    for c in NORMALIZE:        if c not in frame.columns:            continue        mx = gb[c].transform("max")        frame[f"{c}_ratio_to_max"] = np.where(np.abs(mx) > 0, frame[c] / mx, np.nan)        frame[f"{c}_rank"] = gb[c].rank(ascending=False, method="min")    return framedf = add_event_context(df)DROP = set(LEAK) | set(IDS)feat = [c for c in df.columns if c not in DROP]# auto-drop degenerate columns (evaluated on the full set)degen = [c for c in feat         if df[c].isna().all() or df[c].nunique(dropna=True) <= 1]feat = [c for c in feat if c not in degen]print(f"dropped {len(degen)} degenerate columns:")for c in degen: print("   ", c)print(f"\n{len(feat)} features -> X")

## 5. SplitSplit **by event**, never by row — two clusters from the same event must not straddletrain/test, or the model sees the answer. Events are the effective sample size here(~N_events, not N_rows).

In [ ]:
rng = np.random.default_rng(RANDOM)ev  = df[EVT].drop_duplicates().reset_index(drop=True)ev["fold"] = rng.random(len(ev))df2 = df.merge(ev, on=EVT, how="left")train = df2[df2.fold <  0.7].copy()test  = df2[df2.fold >= 0.7].copy()print(f"train {train.groupby(EVT).ngroups:,} events / {len(train):,} rows")print(f"test  {test.groupby(EVT).ngroups:,} events / {len(test):,} rows")

## 6. Train — formulations A, B, C`HistGradientBoosting` handles NaN natively, which matters: `time_chi2_ndf` is NaN for the~32% of clusters with a single timed track, and `hgtd_*` is NaN wherever `RecoVtx_time` ismissing (~8% in VBF, ~55% in Z+jets). **Never impute those with 0** — 0.0 is a legitimate time.

In [ ]:
COMMON = dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=31,              min_samples_leaf=50, l2_regularization=1.0,              early_stopping=True, validation_fraction=0.15,              random_state=RANDOM)models, preds = {}, {}# A — is this the truth-closest cluster?mA = HistGradientBoostingClassifier(**COMMON).fit(train[feat], train["is_best"])test["scoreA"] = mA.predict_proba(test[feat])[:, 1]models["A is_best"] = mA# B — is this cluster inside the 60 ps window? (multi-positive)mB = HistGradientBoostingClassifier(**COMMON).fit(train[feat], train["within60"])test["scoreB"] = mB.predict_proba(test[feat])[:, 1]models["B within60"] = mB# C — regress log1p(|dt|); log compresses a very long tail. Pick the MINIMUM.mC = HistGradientBoostingRegressor(**COMMON).fit(train[feat], np.log1p(train["abs_dt"]))test["scoreC"] = mC.predict(test[feat])models["C reg log|dt|"] = mCfor k, m in models.items():    print(f"{k:16s} iterations used: {m.n_iter_}")

## 7. Results (test set)

In [ ]:
ROWS = [("oracle (ceiling)", "abs_dt", False),        ("TRKPTZ (baseline)", "trkptz_score", True),        ("WAVeS", "waves_score", True),        ("A  is_best",  "scoreA", True),        ("B  within60", "scoreB", True),        ("C  reg |dt|", "scoreC", False)]def table(frame, title):    print(f"\n=== {title}  ({frame.groupby(EVT).ngroups:,} events) ===")    res = {n: select_metric(frame, c, h) for n, c, h in ROWS}    base = res["TRKPTZ (baseline)"][0]    ceil = res["oracle (ceiling)"][0]    print(f"  {'selector':20s} {'core<60ps':>10s} {'median|dt|':>11s} {'vs TRKPTZ':>10s} {'gap closed':>11s}")    print("  " + "-" * 68)    for n, _, _ in ROWS:        cf, md_ = res[n]        d   = "" if n.startswith(("oracle", "TRKPTZ")) else f"{cf-base:+.1f} pts"        gap = "" if n.startswith(("oracle", "TRKPTZ")) or ceil <= base \              else f"{100*(cf-base)/(ceil-base):.0f}%"        print(f"  {n:20s} {cf:9.1f}% {md_:10.1f} ps {d:>10s} {gap:>11s}")    return res_ = table(test, "ALL SAMPLES")for sid, sub in test.groupby("sample_id"):    _ = table(sub, f"sample = {SAMPLE_NAME.get(sid, sid)}")# conditional on the event being recoverable at all -- isolates selection skill_ = table(test[test.recoverable == 1], "RECOVERABLE EVENTS ONLY")

## 8. Cross-topology transfer — the risk that matters mostThis study's central finding is that **VBF and Z+jets prefer opposite behaviour**: the WAVeS jetterm helps VBF (+1.8 pts) and hurts Z+jets (−3.3 pts); LOJO is −10.2 on VBF and −0.4 on Z+jets.A model trained on one topology can therefore learn a rule that is *actively wrong* on another.`sample_id` is excluded from the features, so this matrix measures genuine transfer.Read the off-diagonal: if train-VBF/test-Z+jets collapses, the selector is topology-specific andwe need either a mixed training set (the diagonal-plus row) or a per-topology model.

In [ ]:
def core_of(frame, col, hib=True):    return select_metric(frame, col, hib)[0]sids = sorted(df2["sample_id"].unique())if len(sids) > 1:    hdr = "train / test".rjust(14) + "".join(f"{SAMPLE_NAME.get(s,s):>12s}" for s in sids)    print(hdr + f"{'MIXED':>12s}")    for tr in list(sids) + ["mixed"]:        tr_df = train if tr == "mixed" else train[train.sample_id == tr]        m = HistGradientBoostingClassifier(**COMMON).fit(tr_df[feat], tr_df["is_best"])        label = "mixed" if tr == "mixed" else SAMPLE_NAME.get(tr, tr)        line = f"{label:>14s}"        for te in sids:            sub = test[test.sample_id == te].copy()            sub["p"] = m.predict_proba(sub[feat])[:, 1]            line += f"{core_of(sub,'p'):11.1f}%"        allsub = test.copy(); allsub["p"] = m.predict_proba(allsub[feat])[:, 1]        line += f"{core_of(allsub,'p'):11.1f}%"        print(line)    print("\nreference (TRKPTZ baseline, same test sets):")    line = f"{'TRKPTZ':>14s}"    for te in sids:        line += f"{core_of(test[test.sample_id==te],'trkptz_score'):11.1f}%"    print(line + f"{core_of(test,'trkptz_score'):11.1f}%")else:    print("only one sample loaded — run the other --sample exports to fill this matrix")

## 9. What is the model using?Permutation importance on the physics metric's own quantity (not AUC). Two things to check:- **Is it just relearning TRKPTZ?** `trkptz_score` and its `_ratio_to_max` sitting alone at the  top with everything else flat would mean no real gain — compare the §7 numbers to the baseline.- **Which topology signals matter?** the jet-association block is where VBF and Z+jets disagree.

In [ ]:
sub = test.sample(min(20000, len(test)), random_state=RANDOM)imp = permutation_importance(mA, sub[feat], sub["is_best"], n_repeats=5,                             random_state=RANDOM, scoring="average_precision")order = np.argsort(imp.importances_mean)[::-1][:25]plt.figure(figsize=(7, 8))plt.barh([feat[i] for i in order][::-1], imp.importances_mean[order][::-1],         xerr=imp.importances_std[order][::-1])plt.xlabel("permutation importance (avg precision drop)")plt.title("Model A — top 25 features")plt.tight_layout(); plt.show()print("top 25:")for i in order:    print(f"  {imp.importances_mean[i]:8.5f}  {feat[i]}")

## 10. Residual distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))bins = np.linspace(-400, 400, 81)for name, col, hib in ROWS:    s = test[col].fillna(-np.inf if hib else np.inf)    idx = s.groupby([test[c] for c in EVT], sort=False)    idx = idx.idxmax() if hib else idx.idxmin()    sel = test.loc[idx.to_numpy()]    ax[0].hist(np.clip(sel["delta_t"], bins[0], bins[-1]), bins=bins,               histtype="step", lw=1.8, label=f"{name} ({(sel['abs_dt']<PASS_PS).mean()*100:.1f}%)")    x = np.linspace(0, 300, 200)    ax[1].plot(x, [100*(sel["abs_dt"] < v).mean() for v in x], lw=1.8, label=name)ax[0].set_xlabel("selected time − truth [ps]"); ax[0].set_ylabel("events")ax[0].legend(fontsize=8); ax[0].set_title("Residual")ax[1].axvline(PASS_PS, ls=":", c="k"); ax[1].set_xlabel("|Δt| [ps]")ax[1].set_ylabel("cumulative %"); ax[1].legend(fontsize=8)ax[1].set_title("Fraction within X ps")plt.tight_layout(); plt.show()

## Next steps**Decision gate:** does any of A/B/C beat TRKPTZ on the §7 table, and does the §8 matrix hold upoff-diagonal? If yes → Phase 2. If no, the tabular features are not carrying the signal and theper-track route below is the better bet regardless.1. **Per-track P(HS) model** (highest value). The exporter's `tracks` tree has ~30 rows/event with   `truth_is_hs` — millions of labeled examples instead of ~67k events. Train `P(HS|track)`, then   score clusters as `Σ pT·P(HS)`. That is the *learned* form of exactly what WAVeS hard-codes as   `pT_jet/ΔR`, which is why WAVeS is topology-dependent and this should not be.2. **Deep Sets over clusters** — permutation-invariant, scores each cluster conditioned on its   competitors. The principled version of the §4 hand-built context features.3. **Abstain output** — the ~13% of events with no good cluster are currently unwinnable noise.   A confidence/margin cut gives the "don't trust this vertex time" flag downstream RpT wants.4. **Deployment**: inference runs inside `TTreeProcessorMT` over millions of events (~µs/event,   thread-safe). GBDT or a small MLP via ONNX Runtime is fine; note TMVA is deliberately excluded   from this build (`CMakeLists.txt`), so ONNX is the path.